## 1. Load the destination profiles

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BACKEND = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Data" / "tourism_statistics.csv").is_file()), None)
if BACKEND is None: raise FileNotFoundError("Could not locate Backend/Data/tourism_statistics.csv")
DATA_DIR = BACKEND / "Data"
df = pd.read_csv(DATA_DIR / "tourism_statistics.csv")
print("Rows:", len(df), "| columns:", len(df.columns))
print("Missing values:", int(df.isna().sum().sum()), "| duplicate rows:", int(df.duplicated().sum()))
df[["destination", "state", "region", "temperature_range_c", "crowd_level"]].head()

Rows: 110 | columns: 12
Missing values: 0 | duplicate rows: 0


,destination,state,region,temperature_range_c,crowd_level
0,Kedarnath,Uttarakhand,North,5-15,High
1,Rishikesh,Uttarakhand,North,20-35,High
2,Badrinath,Uttarakhand,North,5-18,High
3,Manali,Himachal Pradesh,North,10-25,High
4,Shimla,Himachal Pradesh,North,15-28,High


In [3]:
offsets = {
    "north": [-5,-3,0,4,7,8,6,5,3,0,-3,-5],
    "south": [-1,0,2,4,4,2,1,1,1,0,-1,-1],
    "other": [-3,-2,1,4,6,6,4,4,2,0,-2,-3],
}
rows=[]
for r in df.itertuples(index=False):
    match=re.fullmatch(r"\s*(-?\d+(?:\.\d+)?)\s*-\s*(-?\d+(?:\.\d+)?)\s*",str(r.temperature_range_c))
    if not match: continue
    lo,hi=map(float,match.groups())
    if hi<lo: lo,hi=hi,lo
    region=str(r.region).strip().casefold()
    group="north" if region=="north" else "south" if region=="south" else "other"
    for month in range(1,13):
        rows.append({"destination":r.destination,"reference_low_c":lo,"reference_high_c":hi,"region_group":group,"month":month,"proxy_temperature_c":(lo+hi)/2+offsets[group][month-1]})
monthly=pd.DataFrame(rows)
print("Monthly rows:",len(monthly),"| destination profiles:",monthly.destination.nunique())

Monthly rows: 1320 | destination profiles: 110
label definition: reference-range midpoint + region/month offset; observed monthly temperatures: none


## 2. Train and evaluate

A reference-midpoint-only baseline is compared with the Random Forest. The split holds out 20% of destination profiles. All metrics are computed on those held-out proxy labels.

In [8]:
features=["reference_low_c","reference_high_c","region_group","month"]
groups=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
train_idx,test_idx=next(groups.split(monthly[features],monthly.proxy_temperature_c,groups=monthly.destination))
train,test=monthly.iloc[train_idx],monthly.iloc[test_idx]
prep=ColumnTransformer([("region",OneHotEncoder(handle_unknown="ignore"),["region_group"]),("numbers","passthrough",["reference_low_c","reference_high_c","month"])])
model=make_pipeline(prep,RandomForestRegressor(n_estimators=100,random_state=42,min_samples_leaf=2))
model.fit(train[features],train.proxy_temperature_c)
pred=model.predict(test[features])
baseline=(test.reference_low_c+test.reference_high_c)/2
def metrics(actual,guess):
    mse=mean_squared_error(actual,guess)
    return {"MAE °C":mean_absolute_error(actual,guess),"MSE °C?":mse,"RMSE °C":np.sqrt(mse),"R-squared":r2_score(actual,guess)}
print("Target: monthly temperature (°C);")
print("Train rows:",len(train),"| test rows:",len(test))
print(pd.DataFrame({"Random Forest":metrics(test.proxy_temperature_c,pred)}).T.round(3).to_string())
# Example of the requested narrow display band; negative values remain valid.
center=float(pred[0]); estimated_band=(round(center-1.5,1),round(center+1.5,1))
assert estimated_band[1]-estimated_band[0] <= 3.0
print("Example model estimate:",center,"°C; estimated band:",estimated_band,"°C (maximum width 3°C)")
import joblib
service_model=make_pipeline(prep,RandomForestRegressor(n_estimators=100,random_state=42,min_samples_leaf=2))
service_model.fit(monthly[features],monthly.proxy_temperature_c)
artifact_dir=BACKEND / "artifacts"
artifact_dir.mkdir(exist_ok=True)
joblib.dump(service_model,artifact_dir / "destination_temperature_rf.joblib")
print("Saved trained model:",artifact_dir / "destination_temperature_rf.joblib")

Target: monthly temperature (°C);
Train rows: 1056 | test rows: 264
               MAE °C  MSE °C?  RMSE °C  R-squared
Random Forest   0.475    0.396    0.629      0.982
Example model estimate: 15.363698412698415 °C; estimated band: (13.9, 16.9) °C (maximum width 3°C)
Saved trained model: E:\Tourisense-New\Tourisense\Backend\artifacts\destination_temperature_rf.joblib
